# Modality robustness — degradation and missing modalities**RQ1** How does beam prediction degrade as each modality becomes unreliable?**RQ2** Can AMBER cope when a modality is missing at inference?Both evaluate **existing checkpoints under modified inputs** — no retraining.That is the right experiment for these questions: they ask how the *deployed*predictor behaves, not whether a model could learn to compensate. Retraining atevery severity would cost tens of GPU-hours and answer a different question.---## Design: which degradations are admissible, and which are notOnly the preprocessed tensors exist at experiment time, so a degradation countsonly if applying it *there* corresponds to a real sensor failure. Inspecting therepresentations rejected several of the obvious candidates.| Modality | Applied | Represents | Why it is admissible ||---|---|---|---|| **Camera** | Gaussian blur | defocus, motion, dirty lens | not affine, survives eq. (10) || | additive noise | low-light sensor noise — scenarios 33/34 are night | not affine || | occlusion (block) | obstruction, raindrop, blocked FOV | not affine || | resolution loss | cheaper sensor, heavy compression | not affine || **LiDAR** | point dropout | sparse returns, rain, low reflectivity | BEV value is count/5, so binomial thinning of the counts is *exactly* "each return independently lost" || **GPS** | positional noise (metres) | consumer GNSS is 1–5 m; this dataset is far more precise | converted through `gps_norm.json`, so severity is physical |### Rejected, with reasons| Rejected | Why ||---|---|| **Camera brightness / contrast** | **Provably a no-op.** Eq. (10) standardises each image by its own mean and std, which is affine-invariant: `a·x + b` → `(a·x + b − a·mean − b)/(a·std)` = the original. Verified to 4e-7. Any lighting experiment here measures nothing. || **Radar noise / map corruption** | The stored tensor is a post-2D-FFT magnitude pair under one joint min-max. Noise in the map domain has no pre-image in the IQ cube, where real noise spreads across the whole map. A curve would measure an artefact. **Radar unreliability is studied through the dataset's own missingness instead** — scenario 34 supplies radar for only ~40 % of its samples. || **LiDAR range truncation** | The BEV is already cropped to the ±50 m ROI holding 99 % of points. || **Fewer GPS observations** | Only two exist; dropping one is the missing-modality experiment. |### What each curve is forCamera is the priority: the ablation showed it is the **only expensive modalitywith substantial incremental value** (+0.104 DBA against LiDAR's +0.023), so itsreliability decides whether escalating to it is ever safe.GPS matters just as much for a different reason: it is the **cheap tier**. Thewhole revised framing rests on GPS alone reaching 83 % of the full model's DBA.If that collapses at realistic localisation error, the framing is fragile. Thisis arguably the single most load-bearing experiment here.LiDAR gets one short curve, not for completeness but because **point density isa candidate gate feature** — we need to know whether it tracks usefulness.

## 1. Configuration

In [ ]:
QUICK_RUN = True        # True: ~200 samples for debugging. False: the real run.SPLIT = "val"BATCH_SIZE = 32WORKERS = 2COLLECT_SENSOR_QUALITY = FalseLIMIT = 200 if QUICK_RUN else Nonefrom pathlib import PathOUT = Path("/kaggle/working/outputs/modality_degradation")(OUT / "plots").mkdir(parents=True, exist_ok=True)MISS = Path("/kaggle/working/outputs/missing_modality")MISS.mkdir(parents=True, exist_ok=True)print(f"QUICK_RUN={QUICK_RUN}  limit={LIMIT}  out={OUT}")

## 2. Environment, data, code

In [ ]:
import osimport socketimport subprocessimport sysfrom pathlib import Pathimport numpy as npimport pandas as pdimport torchprint(f"torch {torch.__version__}   cuda: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_properties(0).name}")def has_internet(host="github.com", port=443, timeout=6):    try:        socket.create_connection((host, port), timeout=timeout).close()        return True    except OSError:        return Falseassert has_internet(), "Internet is off; the code cannot be cloned."

In [ ]:
import shutilINPUT_ROOT = Path("/kaggle/input")SOURCES = ("development", "adaptation", "test")def walk_dirs(base, follow=True):    for dirpath, dirnames, filenames in os.walk(base, followlinks=follow):        yield Path(dirpath), dirnames, filenamesdef find_source(base, name):    for c in (base / name / name, base / name):        if c.is_dir() and any(c.glob("scenario*")):            return c    return Nonedef find_index(base):    hits = []    for c in (base / "index" / "index", base / "index"):        if (c / "samples.csv").exists():            hits.append(c)    for d, _dn, fn in walk_dirs(base):        if "samples.csv" in fn and d not in hits:            hits.append(d)    if not hits:        return None    # prefer the index that sits beside the actual tensors, not a copy inside a    # results dump    score = lambda h: sum(1 for s in SOURCES                          for r in (h.parent, h.parent.parent, h)                          if find_source(r, s) is not None)    return sorted(hits, key=score, reverse=True)[0]INDEX_SRC = find_index(INPUT_ROOT)assert INDEX_SRC is not None, "samples.csv not found; attach the preprocessed dataset"ROOTS = [INDEX_SRC.parent, INDEX_SRC.parent.parent, INDEX_SRC]SOURCE_DIRS = {s: next((d for d in (find_source(r, s) for r in ROOTS) if d), None)               for s in SOURCES}for s in [k for k, v in SOURCE_DIRS.items() if v is None]:    for d, dn, _f in walk_dirs(INPUT_ROOT):        if d.name == s and any(x.startswith("scenario") for x in dn):            SOURCE_DIRS[s] = d            break# stage idempotently: /kaggle/working survives between cell runsSTAGE_DIR = Path("/kaggle/working/data/processed_amber")STAGE_DIR.mkdir(parents=True, exist_ok=True)INDEX_DIR = STAGE_DIR / "index"staged, src_csv = INDEX_DIR / "samples.csv", INDEX_SRC / "samples.csv"if not staged.exists():    shutil.copytree(INDEX_SRC, INDEX_DIR, dirs_exist_ok=True)elif staged.stat().st_size != src_csv.stat().st_size:    shutil.rmtree(INDEX_DIR)    shutil.copytree(INDEX_SRC, INDEX_DIR, dirs_exist_ok=True)for name, src in SOURCE_DIRS.items():    if src is None:        continue    link = STAGE_DIR / name    if link.is_symlink():        try:            same = link.resolve(strict=True) == src.resolve(strict=True)        except OSError:            same = False        if not same:            link.unlink()    elif link.exists():        shutil.rmtree(link)    if not link.exists():        link.symlink_to(src)print(f"index {INDEX_DIR}")

In [ ]:
PROJECT = Path("/kaggle/working/project")if not (PROJECT / "amber" / "model.py").exists():    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/SathishAdithiyaaSV/MultiModal-Beam-Prediction.git", str(PROJECT)],                       capture_output=True, text=True,                       env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})    assert r.returncode == 0, (r.stderr or "")[:400]os.chdir(PROJECT)if str(PROJECT) not in sys.path:    sys.path.insert(0, str(PROJECT))from amber import degrade, qualityfrom amber.ablation import config_label, enabled_mask, parse_modalities, restrict_availabilityfrom amber.config import MODALITIES, TrainConfigfrom amber.dataset import AmberDatasetfrom amber.degrade import Degradationfrom amber.metrics import evaluatefrom amber.predict import load_checkpointfrom amber.train import make_loader, pick_device, to_deviceGFLOPS = {"gps": 0.3869, "gps_radar": 23.5668, "gps_lidar": 23.0530,           "gps_radar_lidar": 46.2328, "gps_image": 48.2398,           "gps_image_lidar": 70.9058, "gps_radar_image": 71.4196,           "gps_radar_image_lidar": 94.0856}def find_checkpoints():    found = {}    for d, _dn, fn in walk_dirs(INPUT_ROOT):        if "best.pt" in fn and d.name in GFLOPS:            found.setdefault(d.name, d / "best.pt")        for f in fn:            if f.endswith(".pt") and f[:-3] in GFLOPS:                found.setdefault(f[:-3], d / f)    return foundCKPTS = find_checkpoints()print(f"{len(CKPTS)}/8 checkpoints: {', '.join(sorted(CKPTS))}")device = pick_device("auto")train_cfg = TrainConfig()

In [ ]:
import gcimport jsonimport time_MODEL_CACHE = {}def get_model(slug):    '''Load once and keep -- several sweeps reuse the same checkpoint.'''    if slug not in _MODEL_CACHE:        if len(_MODEL_CACHE) >= 2:          # keep GPU memory bounded            _MODEL_CACHE.clear()            gc.collect()            if device.type == "cuda":                torch.cuda.empty_cache()        _MODEL_CACHE[slug] = load_checkpoint(CKPTS[slug], device)    return _MODEL_CACHE[slug]@torch.no_grad()def run_eval(slug, split, degradations=(), mask_out=(), limit=None,             return_per_sample=False, collect_features=False):    '''Evaluate one checkpoint, optionally with degraded or masked inputs.    `mask_out` zeroes availability bits at inference -- the eq. (3) mechanism,    not a retrained model. `degradations` perturb the input tensors.    '''    model, cfg = get_model(slug)    mods = parse_modalities(slug.split("_"))    # Sensor-quality features must be computed from the *real* tensors. The    # dataset returns zeros for any modality outside `modalities`, so a cheap    # GPS-only configuration would otherwise yield identically-zero image,    # LiDAR and radar statistics -- which silently collapses feature group C    # onto group B. Load every modality when collecting them; the model is    # still restricted through `keep` below, and availability masking is    # numerically equivalent to not loading (verified: delta logits = 0.0), so    # the logits are unchanged.    ds_mods = (tuple(MODALITIES) if (collect_features and COLLECT_SENSOR_QUALITY)               else mods)    ds = AmberDataset(INDEX_DIR, split, cfg, modalities=ds_mods,                      degradations=list(degradations))    if limit:        from torch.utils.data import Subset        idx = np.linspace(0, len(ds) - 1, min(limit, len(ds))).astype(int)        scen = [ds.scenarios[i] for i in idx]        sids = [ds.sample_ids[i] for i in idx]        ds = Subset(ds, idx.tolist())    else:        scen, sids = ds.scenarios, ds.sample_ids    loader = make_loader(ds, BATCH_SIZE, shuffle=False, workers=WORKERS)    keep = enabled_mask([m for m in mods if m not in mask_out], device) if mask_out \        else enabled_mask(mods, device)    logits, targets, feats = [], [], []    for batch in loader:        raw = to_device(batch, device)        batch = restrict_availability(raw, keep)        out = model(batch, use_cma=False)        logits.append(out.logits.float().cpu())        targets.append(batch["target"].cpu())        if collect_features:            # from `raw`, not `batch`: avail_* should report which sensors            # actually reported, not which this configuration was permitted to            # consume. The latter is constant per configuration and so carries            # no information for the gate.            feats += quality.batch_features(                {k: v.cpu() for k, v in raw.items() if torch.is_tensor(v)},                out.logits.float().cpu(), MODALITIES, COLLECT_SENSOR_QUALITY)    logits, targets = torch.cat(logits), torch.cat(targets)    lab = targets >= 0    m = evaluate(logits[lab], targets[lab], ks=train_cfg.topk,                 delta=train_cfg.dba_delta)    if not (return_per_sample or collect_features):        return m    ranked = logits.topk(3, dim=1).indices.numpy()    true = targets.numpy()[:, None]    best = np.minimum.accumulate(np.abs(ranked - true) / train_cfg.dba_delta,                                 axis=1).clip(max=1.0)    per = pd.DataFrame({"sample_id": sids, "scenario": scen,                        "true_beam": targets.numpy(),                        "pred_beam": ranked[:, 0],                        "correct1": (ranked[:, 0] == true[:, 0]).astype(int),                        "correct3": (ranked[:, :3] == true).any(axis=1).astype(int),                        "dba": (1.0 - best).mean(axis=1)})    if collect_features:        per = pd.concat([per, pd.DataFrame(feats)], axis=1)    return m, per

## 3. RQ1 — degradation curvesEach sweep evaluates one checkpoint whose performance depends on the degradedmodality. GPS is swept on **two** configurations, because whether camera canrescue a degraded cheap tier is itself a routing question.Results are appended to disk after every point, so an interrupted session keepsits progress.

In [ ]:
SWEEPS = [    # (slug, modality, kind)  -- severity grids come from amber.degrade    ("gps_image", "image", "blur"),    ("gps_image", "image", "noise"),    ("gps_image", "image", "occlusion"),    ("gps_image", "image", "resolution"),    ("gps",       "gps",   "position_noise"),    ("gps_image", "gps",   "position_noise"),    ("gps_lidar", "lidar", "dropout"),]SWEEPS = [s for s in SWEEPS if s[0] in CKPTS]rows_path = OUT / f"degradation_{SPLIT}{'_quick' if QUICK_RUN else ''}.csv"done = set()if rows_path.exists():    prev = pd.read_csv(rows_path)    done = set(zip(prev.slug, prev.modality, prev.kind, prev.severity))    rows = prev.to_dict("records")    print(f"resuming: {len(rows)} points already measured")else:    rows = []for slug, modality, kind in SWEEPS:    for sev in degrade.grid_for(kind):        key = (slug, modality, kind, sev)        if key in done:            continue        degs = [] if sev == degrade.grid_for(kind)[0] else [Degradation(modality, kind, sev)]        t0 = time.time()        m = run_eval(slug, SPLIT, degradations=degs, limit=LIMIT)        rows.append({"slug": slug, "config": config_label(parse_modalities(slug.split("_"))),                     "modality": modality, "kind": kind, "severity": sev,                     "Top-1": m["top1"], "Top-3": m["top3"], "Top-5": m["top5"],                     "DBA": m["dba"], "n": m["n"], "seconds": round(time.time() - t0, 1)})        pd.DataFrame(rows).to_csv(rows_path, index=False)        print(f"  {slug:<12} {kind:<15} sev={sev:<7g} "              f"Top-1 {m['top1']:.4f}  DBA {m['dba']:.4f}  [{rows[-1]['seconds']}s]")deg = pd.DataFrame(rows)print(f"\n{len(deg)} points -> {rows_path}")

### Degradation table and where each modality collapses

In [ ]:
for (slug, kind), g in deg.groupby(["slug", "kind"], sort=False):    g = g.sort_values("severity", key=lambda s: s.abs() if kind != "dropout" else -s)    base = g.iloc[0]    print(f"\n{g.iloc[0]['config']}  --  {kind}")    print(f"{'severity':>10}{'Top-1':>9}{'DBA':>9}{'ΔDBA':>9}{'% of intact DBA':>18}")    for _, r in g.iterrows():        print(f"{r.severity:>10g}{r['Top-1']:>9.4f}{r.DBA:>9.4f}"              f"{r.DBA - base.DBA:>+9.4f}{r.DBA / base.DBA * 100:>17.1f}%")# where does each sweep lose a quarter of its intact DBA?print("\n\ntipping points -- first severity costing >25% of intact DBA:")for (slug, kind), g in deg.groupby(["slug", "kind"], sort=False):    g = g.sort_values("severity", key=lambda s: -s if kind == "dropout" else s)    base = g.iloc[0].DBA    hit = g[g.DBA < 0.75 * base]    where = f"{hit.iloc[0].severity:g}" if len(hit) else "never in range"    print(f"  {g.iloc[0]['config']:<24} {kind:<15} -> {where}")

## 4. RQ2 — missing modalitiesTwo different questions, separated.**(a) Does AMBER's eq. (3) mask actually work?** Take the model trained on allfour modalities, mask one out at inference, and compare against the model*independently trained* without it. If masking is far worse, themissing-modality mechanism the paper is built on is not delivering — a directtest of its central claim.**(b) Natural missingness.** Radar is genuinely absent for ~60 % of scenario-34samples. Those rows are reported separately from synthetically masked ones, sothe two are never conflated.Subsets are chosen, not enumerated: each single modality, plus the three pairsthat leave a usable configuration. Dropping GPS is included because it is thecheap tier.

In [ ]:
FULL = "gps_radar_image_lidar"SUBSETS = [(), ("image",), ("lidar",), ("radar",), ("gps",),           ("image", "radar"), ("image", "lidar"), ("radar", "lidar")]miss_rows = []if FULL in CKPTS:    for drop in SUBSETS:        remaining = [m for m in ("gps", "radar", "image", "lidar") if m not in drop]        m = run_eval(FULL, SPLIT, mask_out=drop, limit=LIMIT)        miss_rows.append({            "dropped": "none" if not drop else "+".join(drop),            "remaining": "+".join(remaining),            "source": "masked full model",            "Top-1": m["top1"], "Top-3": m["top3"], "Top-5": m["top5"], "DBA": m["dba"],            "GFLOPs": GFLOPS[FULL],   # masking does not save compute: see note below            "n": m["n"]})        print(f"  drop {str(drop):<22} Top-1 {m['top1']:.4f}  DBA {m['dba']:.4f}")miss = pd.DataFrame(miss_rows)miss.to_csv(MISS / f"masked_full_model_{SPLIT}.csv", index=False)print(f"\n-> {MISS}/masked_full_model_{SPLIT}.csv")print("\nNote: masking the full model does NOT reduce its cost -- the encoders still")print("run unless skip_unavailable_encoders is set. Cost savings come from using a")print("smaller configuration, which is what the ablation measured.")

### Masking vs independent training — does the mask mechanism hold up?

In [ ]:
# the ablation trained a model for each subset; compare like with likeABL = {"gps": "GPS", "gps_radar": "GPS + Radar", "gps_lidar": "GPS + LiDAR",       "gps_image": "GPS + Camera", "gps_radar_lidar": "GPS + Radar + LiDAR",       "gps_radar_image": "GPS + Radar + Camera", "gps_image_lidar": "GPS + Camera + LiDAR",       "gps_radar_image_lidar": "GPS + Radar + Camera + LiDAR"}cmp_rows = []for _, r in miss.iterrows():    want = set(r.remaining.split("+"))    twin = next((s for s in ABL if set(s.split("_")) == want and s in CKPTS), None)    if twin is None:        continue    t = run_eval(twin, SPLIT, limit=LIMIT)    cmp_rows.append({"modalities": r.remaining,                     "masked full model DBA": r.DBA,                     "independently trained DBA": t["dba"],                     "gap": r.DBA - t["dba"],                     "masked Top-1": r["Top-1"], "trained Top-1": t["top1"]})cmp = pd.DataFrame(cmp_rows).sort_values("gap")cmp.to_csv(MISS / f"masking_vs_training_{SPLIT}.csv", index=False)print(cmp.round(4).to_string(index=False))if len(cmp):    print(f"\nmean gap {cmp.gap.mean():+.4f} DBA")    print("Negative means masking the full model is WORSE than training without the")    print("modality -- i.e. the mask leaves the model depending on something absent.")    print("Near zero means the eq. (3) mechanism genuinely handles the absence.")

### Natural vs synthetic missingness

In [ ]:
samples = pd.read_csv(INDEX_DIR / "samples.csv")v = samples[samples.split == SPLIT]nat = v.groupby("scenario")[["m_image", "m_lidar", "m_radar", "m_gps"]].mean().round(3)print("naturally available fraction per scenario on this split:")print(nat.to_string())print("\nRadar is naturally absent for a large share of scenario 34. Splitting the")print("best configuration's per-sample results on that real mask separates")print("'sensor genuinely absent' from 'we removed it', which the table above mixes.")best = "gps_radar_image_lidar" if FULL in CKPTS else sorted(CKPTS)[0]_m, per = run_eval(best, SPLIT, limit=LIMIT, return_per_sample=True)per = per.merge(v[["sample_id", "m_radar"]], on="sample_id", how="left")nat_rows = [{"radar_naturally_present": bool(k), "n": len(g),             "Top-1": g.correct1.mean(), "DBA": g.dba.mean()}            for k, g in per.groupby("m_radar")]nat_df = pd.DataFrame(nat_rows)nat_df.to_csv(MISS / f"natural_missingness_{SPLIT}.csv", index=False)print()print(nat_df.round(4).to_string(index=False))

## 5. Plots

In [ ]:
import matplotlib.pyplot as pltgroups = list(deg.groupby(["slug", "kind"], sort=False))ncol = min(4, len(groups))nrow = (len(groups) + ncol - 1) // ncolfig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.4 * nrow), squeeze=False)for ax, ((slug, kind), g) in zip(axes.flat, groups):    g = g.sort_values("severity")    x = g.severity if kind != "dropout" else 1 - g.severity    ax.plot(x, g.DBA, marker="o", label="DBA")    ax.plot(x, g["Top-1"], marker="s", ls="--", label="Top-1")    ax.set(title=f"{g.iloc[0]['config']}\n{kind}",           xlabel="severity" if kind != "dropout" else "fraction of points lost",           ylim=(0, 1))    ax.grid(alpha=.3)    ax.legend(frameon=False, fontsize=8)for ax in axes.flat[len(groups):]:    ax.set_visible(False)plt.tight_layout()plt.savefig(OUT / "plots" / "degradation_curves.png", dpi=150)plt.show()print(f"-> {OUT}/plots/degradation_curves.png")

## 6. Save

In [ ]:
meta = {"quick_run": QUICK_RUN, "split": SPLIT, "limit": LIMIT,        "sweeps": [{"slug": s, "modality": m, "kind": k,                    "grid": degrade.grid_for(k)} for s, m, k in SWEEPS],        "rejected_degradations": degrade.REJECTED,        "missing_subsets": [list(s) for s in SUBSETS],        "checkpoints_used": sorted(CKPTS)}(OUT / "config.json").write_text(json.dumps(meta, indent=2))for p in sorted(list(OUT.rglob("*")) + list(MISS.rglob("*"))):    if p.is_file():        print(f"  {p}  ({p.stat().st_size/1e3:.1f} KB)")